# Partie 1 — Exploration et nettoyage des données

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
print(np.__version__)  # doit afficher 1.26.4

2.4.4


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from mlxtend.frequent_patterns import apriori, association_rules

In [3]:
#chargement des données 
df = pd.read_csv(r"C:\Users\dell\Downloads\archive\online_retail_II.csv")

In [4]:
#exloration des donnees comme 1er  étape 
df.info()
df.describe()


<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 136.6 MB


,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


In [5]:
#des valeurs négatifs quantité et prix ce n'est pas un erreur mais signifie des produits retour / transaction annuler il faux supprimer 

In [6]:
df = df.dropna(subset=['Customer ID'])  # supp id null

In [7]:
df = df.drop_duplicates()

In [8]:
df = df[df['Quantity'] > 0] #garder les valeurs positive
df = df[df['Price'] > 0]

In [9]:
print(f"Cleaned dataset: {df.shape}")
print(f"Rows removed: {1067371 - df.shape[0]:,}")


Cleaned dataset: (779425, 8)
Rows removed: 287,946


---
# Partie 2 — Construction des indicateurs comportementaux

In [21]:
# convertir InvoiceDate en datetime pour calculer la récence
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Customer ID'] = df['Customer ID'].astype(int)

In [22]:
# montant total par ligne
df['TotalPrice'] = df['Quantity'] * df['Price']

In [25]:
# date de référence = lendemain de la dernière transaction
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# récence, fréquence, dépense totale
rfm = df.groupby('Customer ID').agg(
    Recence   = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequence = ('Invoice',     'nunique'),
    Depense   = ('TotalPrice',  'sum')
).reset_index()

rfm.head(10)

,Customer ID,Recence,Frequence,Depense
0,12346,326,12,77556.46
1,12347,2,8,4921.53
2,12348,75,5,2019.40
3,12349,19,4,4428.69
4,12350,310,1,334.40
5,12351,375,1,300.93
6,12352,36,10,2849.84
7,12353,204,2,406.76
8,12354,232,1,1079.40
9,12355,214,2,947.61


In [26]:
# taille moyenne du panier (nb d'articles par commande)
taille_panier = df.groupby(['Customer ID', 'Invoice'])['Quantity'].sum().reset_index()
taille_panier.columns = ['Customer ID', 'Invoice', 'TailleCommande']

taille_moy = taille_panier.groupby('Customer ID')['TailleCommande'].mean().reset_index()
taille_moy.columns = ['Customer ID', 'TailleMoyPanier']

taille_moy.head()

,Customer ID,TailleMoyPanier
0,12346,6190.416667
1,12347,370.875000
2,12348,542.800000
3,12349,406.000000
4,12350,197.000000


In [27]:
# diversité des achats (nb de produits distincts achetés)
diversite = df.groupby('Customer ID')['StockCode'].nunique().reset_index()
diversite.columns = ['Customer ID', 'DiversiteProduits']

diversite.head()

,Customer ID,DiversiteProduits
0,12346,27
1,12347,126
2,12348,25
3,12349,138
4,12350,17


In [33]:
# fusion de tous les indicateurs
customer_features = rfm \
    .merge(taille_moy, on='Customer ID') \
    .merge(diversite,  on='Customer ID')

print(f"Nombre de clients : {len(customer_features)}")
customer_features.head(10)

Nombre de clients : 5878


,Customer ID,Recence,Frequence,Depense,TailleMoyPanier,DiversiteProduits
0,12346,326,12,77556.46,6190.416667,27
1,12347,2,8,4921.53,370.875000,126
2,12348,75,5,2019.40,542.800000,25
3,12349,19,4,4428.69,406.000000,138
4,12350,310,1,334.40,197.000000,17
5,12351,375,1,300.93,261.000000,21
6,12352,36,10,2849.84,72.400000,70
7,12353,204,2,406.76,106.000000,23
8,12354,232,1,1079.40,530.000000,58
9,12355,214,2,947.61,271.500000,35


In [35]:
customer_features.describe().round(2)

,Customer ID,Recence,Frequence,Depense,TailleMoyPanier,DiversiteProduits
count,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00
mean,15315.31,201.33,6.29,2955.90,247.56,81.99
std,1715.57,209.34,13.01,14440.85,1424.48,116.48
min,12346.00,1.00,1.00,2.95,1.00,1.00
25%,13833.25,26.00,1.00,342.28,91.00,19.00
50%,15314.50,96.00,3.00,867.74,153.58,45.00
75%,16797.75,380.00,7.00,2248.30,256.93,103.00
max,18287.00,739.00,398.00,580987.04,87167.00,2550.00


---
# Partie 3 — Segmentation non supervisée

Nous testons trois valeurs de k (k=2, k=3, k=4) et nous comparons les résultats à l'aide de métriques d'évaluation pour choisir la meilleure segmentation.

In [ ]:
# normalisation des indicateurs avant clustering
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

cols = ['Recence', 'Frequence', 'Depense', 'TailleMoyPanier', 'DiversiteProduits']
scaler = MinMaxScaler()
X = scaler.fit_transform(customer_features[cols])

In [ ]:
# test des trois valeurs de k et calcul des métriques
K_values = [2, 3, 4]
resultats = []

for k in K_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    resultats.append({
        'k'                     : k,
        'Inertie'               : round(km.inertia_, 2),
        'Silhouette (↑)'        : round(silhouette_score(X, labels), 4),
        'Davies-Bouldin (↓)'    : round(davies_bouldin_score(X, labels), 4),
        'Calinski-Harabasz (↑)' : round(calinski_harabasz_score(X, labels), 2)
    })

comp_k = pd.DataFrame(resultats).set_index('k')
print(comp_k)

In [ ]:
# visualisation des métriques de comparaison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar([str(k) for k in K_values], comp_k['Silhouette (↑)'], color='steelblue')
axes[0].set_title('Silhouette (plus élevé = meilleur)')
axes[0].set_xlabel('k')

axes[1].bar([str(k) for k in K_values], comp_k['Davies-Bouldin (↓)'], color='darkorange')
axes[1].set_title('Davies-Bouldin (plus bas = meilleur)')
axes[1].set_xlabel('k')

axes[2].bar([str(k) for k in K_values], comp_k['Calinski-Harabasz (↑)'], color='green')
axes[2].set_title('Calinski-Harabasz (plus élevé = meilleur)')
axes[2].set_xlabel('k')

plt.suptitle('Comparaison des métriques pour k = 2, 3, 4', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# choix du k final
# k=2 donne les meilleures métriques statistiques (Silhouette et Davies-Bouldin)
# mais k=4 est retenu pour des raisons de logique métier :
# il permet de distinguer 4 profils actionnables pour l'entreprise :
#   → clients fidèles, clients occasionnels, clients inactifs, nouveaux clients
# une segmentation en 2 groupes est trop grossière pour piloter des actions marketing ciblées
k_final = 4
km_final = KMeans(n_clusters=k_final, random_state=42, n_init=10)
customer_features['Segment'] = km_final.fit_predict(X)

print('Répartition des clients par segment :')
print(customer_features['Segment'].value_counts().sort_index())

In [ ]:
# visualisation PCA 2D des 4 segments
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

couleurs = ['steelblue', 'darkorange', 'green', 'crimson']
labels_pca = ['Clients occasionnels', 'Clients fidèles à forte dépense',
              'Clients perdus', 'Nouveaux clients']

plt.figure(figsize=(9, 6))
for seg in range(4):
    masque = customer_features['Segment'] == seg
    plt.scatter(X_pca[masque, 0], X_pca[masque, 1],
                c=couleurs[seg], label=labels_pca[seg], alpha=0.5, s=20)

centroides_2d = pca.transform(km_final.cluster_centers_)
plt.scatter(centroides_2d[:, 0], centroides_2d[:, 1],
            c='black', marker='X', s=200, label='Centroïdes')

var_exp = pca.explained_variance_ratio_ * 100
plt.xlabel(f'PC1 ({var_exp[0]:.1f}% variance expliquée)')
plt.ylabel(f'PC2 ({var_exp[1]:.1f}% variance expliquée)')
plt.title('Visualisation des segments clients — PCA 2D (k=4)')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

print(f'Variance totale expliquée : {sum(var_exp):.1f}%')
for seg in range(4):
    n = (customer_features['Segment'] == seg).sum()
    print(f'  {labels_pca[seg]} : {n} clients ({n/len(customer_features)*100:.1f}%)')

In [ ]:
# profil moyen de chaque segment
profils = customer_features.groupby('Segment')[cols].mean().round(2)
print(profils)

In [ ]:
profils.T.plot(kind='bar', figsize=(10, 5))
plt.title('Profil moyen par segment (k=4 retenu — logique métier)')
plt.ylabel('Valeur moyenne normalisée')
plt.xticks(rotation=30)
plt.legend(title='Segment')
plt.tight_layout()
plt.show()

In [ ]:
# étiquettes métier des 4 segments
labels_segments = {
    0: 'Clients occasionnels',
    1: 'Clients fidèles à forte dépense',
    2: 'Clients perdus',
    3: 'Nouveaux clients'
}
customer_features['ProfilClient'] = customer_features['Segment'].map(labels_segments)
customer_features[['Customer ID', 'Segment', 'ProfilClient']].head(10)

---
# Partie 4 — Règles d'association avec FP-Growth

Nous utilisons **FP-Growth** pour rechercher les associations fréquentes entre produits, puis nous filtrons les résultats pour ne garder que ceux à intérêt métier réel.

In [ ]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

# matrice panier binaire — uniquement le Royaume-Uni
df_uk = df[df['Country'] == 'United Kingdom']

basket = df_uk.groupby(['Invoice', 'Description'])['Quantity'].sum().unstack(fill_value=0)
basket_bin = (basket > 0).astype(bool)

print(f'Matrice panier : {basket_bin.shape[0]} factures × {basket_bin.shape[1]} produits')

In [ ]:
# extraction des itemsets fréquents avec FP-Growth
frequent_itemsets = fpgrowth(basket_bin, min_support=0.02, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False)

print(f"Nombre d'itemsets fréquents trouvés : {len(frequent_itemsets)}")
frequent_itemsets.head(10)

In [ ]:
# génération des règles d'association
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1)
rules = rules.sort_values('lift', ascending=False)

print(f"Nombre total de règles générées : {len(rules)}")
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

In [ ]:
# filtrage — seuils d'intérêt métier
# support > 0.02  : présent dans au moins 2% des factures
# confidence > 0.5 : B acheté dans plus de 50% des cas où A est acheté
# lift > 2         : relation 2x plus probable que le hasard
rules_filtrees = rules[
    (rules['support']    > 0.02) &
    (rules['confidence'] > 0.5)  &
    (rules['lift']       > 2)
].copy()

print(f"Règles pertinentes après filtrage : {len(rules_filtrees)}")
rules_filtrees[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

In [ ]:
# visualisation support vs confiance (taille des points = lift)
plt.figure(figsize=(8, 5))
plt.scatter(
    rules_filtrees['support'],
    rules_filtrees['confidence'],
    s=rules_filtrees['lift'] * 10,
    alpha=0.5,
    color='steelblue'
)
plt.xlabel('Support')
plt.ylabel('Confiance')
plt.title("FP-Growth — Support vs Confiance (taille = Lift)")
plt.tight_layout()
plt.show()

---
# Partie 5 — Comparaison des résultats et recommandations

Nous comparons les règles d'association globales avec les règles obtenues sur chaque segment client pour vérifier si les associations sont homogènes ou spécifiques à certains profils.

In [ ]:
# rattacher le segment de chaque client aux transactions
df_seg = df.merge(
    customer_features[['Customer ID', 'Segment', 'ProfilClient']],
    on='Customer ID', how='left'
)

In [ ]:
# fonction : règles FP-Growth pour un segment donné
def regles_segment(segment_id, min_support=0.02, min_confidence=0.5, min_lift=2):
    sous_df = df_seg[df_seg['Segment'] == segment_id]
    basket_s = sous_df.groupby(['Invoice', 'Description'])['Quantity'].sum().unstack(fill_value=0)
    basket_s = (basket_s > 0).astype(bool)

    fi = fpgrowth(basket_s, min_support=min_support, use_colnames=True)
    if fi.empty:
        print(f'Segment {segment_id} : aucun itemset fréquent.')
        return pd.DataFrame()

    r = association_rules(fi, metric='lift', min_threshold=min_lift)
    r = r[r['confidence'] > min_confidence].sort_values('lift', ascending=False)
    return r[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

# top 5 règles par segment
for seg in sorted(customer_features['Segment'].unique()):
    label = labels_segments.get(seg, f'Segment {seg}')
    print(f'\n=== {label} ===')
    r = regles_segment(seg)
    print(r.head(5).to_string(index=False) if not r.empty else 'Aucune règle.')

In [ ]:
# comparaison du nombre de règles globales vs par segment
comparaison = {'Global': len(rules_filtrees)}

for seg in sorted(customer_features['Segment'].unique()):
    label = labels_segments.get(seg, f'Segment {seg}')
    r = regles_segment(seg)
    comparaison[label] = len(r)

comp_df = pd.DataFrame.from_dict(comparaison, orient='index', columns=['Nb règles pertinentes'])
print(comp_df)

comp_df.plot(kind='bar', legend=False, figsize=(8, 4), color='steelblue')
plt.title('Nombre de règles pertinentes — global vs par segment')
plt.ylabel('Nombre de règles')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# importance des variables dans la segmentation — RandomForest
# remplace SHAP : interprétation simple et directe sans dépendance externe
from sklearn.ensemble import RandomForestClassifier

X_df = pd.DataFrame(X, columns=cols)
y    = customer_features['Segment'].values

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_df, y)

importances = pd.Series(rf.feature_importances_, index=cols).sort_values(ascending=True)

plt.figure(figsize=(7, 4))
importances.plot(kind='barh', color='steelblue')
plt.title('Importance des variables dans la segmentation (RandomForest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print(importances.sort_values(ascending=False))

## Recommandations pour l'entreprise

Les recommandations s'appuient sur les 4 profils clients obtenus et sur l'importance des variables identifiée par le RandomForest.

**1. Fidéliser les clients fidèles à forte dépense**  
Ce segment génère l'essentiel du chiffre d'affaires. La *Dépense* et la *Fréquence* sont les variables les plus déterminantes. Un programme de fidélité exclusif (remises VIP, accès anticipé) permet de conserver et renforcer ce segment.

**2. Réactiver les clients perdus**  
La *Récence* élevée est le signal principal de ce profil. Une campagne de réactivation ciblée (offre limitée dans le temps) doit être lancée avant 6 mois d'inactivité.

**3. Convertir les clients occasionnels**  
Faible fréquence et faible dépense caractérisent ce segment. Des promotions personnalisées et des rappels automatiques peuvent augmenter leur engagement.

**4. Accompagner les nouveaux clients**  
Ce segment a un fort potentiel. Un parcours d'onboarding (email de bienvenue, offre premier achat, recommandations personnalisées) améliore leur rétention.

**5. Exploiter les règles FP-Growth par segment**  
Les associations entre produits varient selon les profils. Les recommandations de cross-sell doivent être construites par segment pour être pertinentes et éviter de noyer les clients avec des suggestions non adaptées.